# ニュース記事を 3 行に要約する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_summarization/three_line_summary.ipynb)

NLP クックブック 28-05 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

記事を「3 つの箇条書き」にまとめたい。形式が崩れた出力を検出して、決まった形に整えたい。

In [1]:
import re
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデル (LLM) の章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat(messages, max_new_tokens=160):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

INSTRUCTION = ("次の記事を 3 行に要約してください。\n"
               "- 必ず 3 行で、各行は「・」で始める\n- 1 行は 35 字以内の 1 文にする\n- 記事に書かれていないことは書かない\n- 3 行以外の文章 (前置きや補足) は出力しない")
EXAMPLE_IN = ("気象庁は 10 日、関東甲信地方が梅雨入りしたとみられると発表した。平年より 3 日遅く、昨年より 2 日早い。"
              "前線の影響で、11 日にかけて太平洋側を中心に大雨となるおそれがある。同庁は土砂災害や低い土地の浸水に警戒を呼びかけている。")
EXAMPLE_OUT = "・関東甲信地方が梅雨入りしたとみられる\n・平年より 3 日遅く、昨年より 2 日早い\n・11 日にかけて太平洋側で大雨のおそれ"

def parse(output):
    """「・」で始まる行だけを取り出す。番号付きや「-」で書かれても拾う"""
    lines = [re.sub(r"^\s*(?:[・\-*●]|\d+[.．)])\s*", "", l).strip() for l in output.splitlines() if re.match(r"^\s*(?:[・\-*●]|\d+[.．)])", l)]
    return [l for l in lines if l]

def three_lines(text, max_retries=1):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{EXAMPLE_IN}"}, {"role": "assistant", "content": EXAMPLE_OUT},   # 出力例を 1 つ見せる
                {"role": "user", "content": f"{INSTRUCTION}\n\n{text}"}]
    for attempt in range(max_retries + 1):
        output = chat(messages)
        lines = parse(output)
        if len(lines) == 3:
            return lines, attempt
        messages += [{"role": "assistant", "content": output}, {"role": "user", "content": "形式が違います。「・」で始まる 3 行だけを出力し直してください。"}]
    return (lines + ["(要約できませんでした)"] * 3)[:3], attempt      # それでも駄目なら、先頭 3 行に切り詰める・埋める

articles = [
    ("市は 14 日、来年 4 月から市内の全小中学校で給食費を無償化すると発表した。対象は約 1 万 2 千人で、年間の費用は約 6 億円を見込む。"
     "財源には、ふるさと納税の寄付金と、公共施設の統廃合で生じる削減分を充てる。市長は記者会見で「子育て世帯の負担を減らし、若い世代に選ばれるまちにしたい」と述べた。"
     "一方、市議会の一部からは「恒久的な財源の裏付けが弱い」との指摘が出ている。市は来月の議会に関連する条例案を提出する方針だ。"),
    ("大学の研究チームは、スマートフォンのカメラで撮影した葉の画像から、稲の病気を判定する人工知能を開発した。約 5 万枚の画像で学習させ、8 種類の病気を 9 割以上の精度で見分けられるという。"
     "農家が専門家に相談する前に、その場で病気の見当を付けられるのが利点だ。チームは来年度、県内の農協と協力して実証実験を行い、3 年後の実用化を目指す。"
     "課題は、照明や撮影角度による精度のばらつきで、今後は撮影方法の案内機能も加える予定だ。"),
]
for text in articles:
    lines, retries = three_lines(text)
    print(f"(再試行 {retries} 回)")
    for l in lines:
        print(f"  ・{l}  ({len(l)} 字)")

(再試行 0 回)
  ・横浜市は市内の全小中学校で給食費を無償化を発表  (23 字)
  ・対象は約 1 万 2 千人、年間費用約 6 億円  (24 字)
  ・財源はふるさと納税寄付金と公共施設削減分、議会で条例案提出へ  (30 字)
(再試行 0 回)
  ・研究チームがスマートフォンのカメラで撮影した写真から稲の病気を判定する人工知能を開発  (42 字)
  ・約 5 万枚の画像で学習させ、8 種類の病気を 9 割以上の精度で見分けられる  (39 字)
  ・来年度から県内農協と協力し実証実験、3 年後の実用化を目指す  (30 字)


- 形式の指定は、箇条書きの条件として具体的に書き、出力例を 1 つ対話の形で見せると安定する。
- それでも形式が崩れることはある。「解析して検証 → 駄目なら再生成 → 最後は機械的に整形」と、必ず決まった形で返る仕組みにしておく。
- 1 つ目の記事では、原文に「市」としか書かれていないのに「横浜市」と書いている。「記事に書かれていないことは書かない」と指示しても、小さいモデルは固有名詞を作ってしまうことがある。
- 3 行要約は、1 行目に最も重要な事実、2 行目以降に補足、という構成が読みやすい。順序も指示できる。
- 文字数の上限は守られないことが多い。厳密に必要なら、行ごとに検査して長い行だけ言い換えさせる。